# Understanding Backpropagation

Evaluate a small graph of operations by hand, then derive and combine its local derivatives to compute several partial derivatives via the chain rule -- the manual version of backpropagation.


In [ ]:
%matplotlib inline
import utils
import numpy as np

# -- start --
## This code will be removed by the course compiler
import shutil

def _generate_inputs(seed):
    rng = np.random.RandomState(seed)
    data = {'x': [], 'w': []}
    for _ in range(100):
        data['x'].append(rng.randn(2))
        data['w'].append(rng.randn(4))
    return data

utils.checker.save_npz(_generate_inputs(0), str(utils.checker.get_data_dir() / 'inputs_public.npz'))
utils.checker.save_npz(_generate_inputs(42), str(utils.checker.get_data_dir() / 'inputs_private.npz'))

if not utils.checker.is_valid_npz('xinputs.npz'):
    shutil.copyfile(str(utils.checker.get_data_dir() / 'inputs_public.npz'), 'xinputs.npz')
# -- end --

inputs_flat = utils.checker.load_answer('xinputs.npz')
inputs = [
    {'x': inputs_flat[f'x -> {i}'], 'w': inputs_flat[f'w -> {i}']}
    for i in range(100)
]

correct_answer = utils.checker.load_answer(
    'yanswer.npz',
    missing_ok=True,
)

answer = {}


# Backpropagation Practice

You are given a graph of operations:

<img src="imgs/task_2_4_graph_02.png" width="400" height="300" />

# Task 1

Write a function that calculates the value of this graph for any given input vector `x` and set of coefficients $b_1, b_2, c_1, c_2$ packed as vector of weights `w`.
In this function also return all the $z$ values.

In [ ]:
# Edit this cell in task.ipynb.
## YOU CAN DEFINE ANY FUNCTIONS YOU NEED ##
## YOUR CODE HERE ##
# -- start --
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)
# -- end --

def graph_value(x, w):
    y = x
    z = [0.] * 9  # fallback so callers that unpack z still work
    ## YOUR CODE HERE ##
    # -- start --
    x_1 = x[0]
    x_2 = x[1]

    b_1 = w[0]
    b_2 = w[1]
    c_1 = w[2]
    c_2 = w[3]

    z_1 = x_1 + b_1
    z_2 = x_2 + b_2
    z_3 = sigmoid(z_1)
    z_4 = sigmoid(z_2)
    z_5 = tanh(z_2)
    z_6 = z_5 * c_2
    z_7 = z_1 * z_4
    z_8 = z_7 * c_1
    z_9 = z_3 * z_6
    z = [z_1, z_2, z_3, z_4, z_5, z_6, z_7, z_8, z_9]
    y = z_8 + z_9
    # -- end --
    return y, z


In [ ]:
answer['graph_value'] = [graph_value(**input) for input in inputs]


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'graph_value'))

# Task 2

In this graph, find all direct derivatives of all the values over $z$ or $x$ or $w$.

For example, if $y = z_8 + z_9$, you need to find only $\partial_{z_9} y$ and $\partial_{z_8} y$ for that operation.

You have to do that for all the intermediate signals.

Write the result in the form of a dictionary, for example, $\partial_{z_9} y$:

```
['dy_dz9'] = 1
```

In [ ]:
# Edit this cell in task.ipynb.
def graph_derivatives(x, w):
    res = {}
    ## YOUR CODE HERE ##
    # -- start --
    y, z = graph_value(x, w)

    z1, z2, z3, z4, z5, z6, z7, z8, z9 = z
    b1, b2, c1, c2 = w
    x1, x2 = x
    
    # y derivatives
    res['dy_dz9'] = 1
    res['dy_dz8'] = 1

    # z9 derivatives
    res['dz9_dz6'] = z3
    res['dz9_dz3'] = z6

    # z8 derivatives
    res['dz8_dc1'] = z7
    res['dz8_dz7'] = c1

    # z7 derivatives
    res['dz7_dz1'] = z[3]
    res['dz7_dz4'] = z[0]

    # z6 derivatives
    res['dz6_dc2'] = z[4]
    res['dz6_dz5'] = w[3]

    # z5 derivatives
    res['dz5_dz2'] = (1 - z[4]) * (1 + z[4])

    # z4 derivatives
    res['dz4_dz2'] = z[3] * (1 - z[3])

    # z3 derivatives
    res['dz3_dz1'] = z[2] * (1 - z[2])

    # z2 derivatives
    res['dz2_dx2'] = 1
    res['dz2_db2'] = 1

    # z1 derivatives
    res['dz1_dx1'] = 1
    res['dz1_db1'] = 1
    # -- end --

    return res


In [ ]:
answer['graph_derivatives'] = [graph_derivatives(**input) for input in inputs]


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'graph_derivatives'))

# Task 3

Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_1} y$
- calculate that derivative

In [ ]:
# Edit this cell in task.ipynb.
def dy_dc1(x, w):
    selected_derivs = {}
    dy_dc1 = 0

    ## YOUR CODE HERE ##
    # -- start --
    y, z = graph_value(x, w)
    derivs = graph_derivatives(x, w)

    selected_derivs['dy_dz8'] = derivs['dy_dz8']
    selected_derivs['dz8_dc1'] = derivs['dz8_dc1']

    dy_dc1 = derivs['dy_dz8'] * derivs['dz8_dc1']
    # -- end --

    return selected_derivs, dy_dc1


In [ ]:
answer['dy_dc1'] = [dy_dc1(**input) for input in inputs]


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'dy_dc1'))

# Task 4
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_2} y$
- calculate that derivative

In [ ]:
# Edit this cell in task.ipynb.
def dy_dc2(x, w):
    selected_derivs = {}
    dy_dc2 = 0

    ## YOUR CODE HERE ##
    # -- start --
    y, z = graph_value(x, w)
    derivs = graph_derivatives(x, w)

    selected_derivs['dy_dz9'] = derivs['dy_dz9']
    selected_derivs['dz9_dz6'] = derivs['dz9_dz6']
    selected_derivs['dz6_dc2'] = derivs['dz6_dc2']

    dy_dc2 = derivs['dy_dz9'] * derivs['dz9_dz6'] * derivs['dz6_dc2']
    # -- end --

    return selected_derivs, dy_dc2


In [ ]:
answer['dy_dc2'] = [dy_dc2(**input) for input in inputs]


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'dy_dc2'))

# Task 5
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_1} y$
- calculate that derivative

In [ ]:
# Edit this cell in task.ipynb.
def dy_db1(x, w):
    selected_derivs = {}
    dy_db1 = 0

    ## YOUR CODE HERE ##
    # -- start --
    # y z8 z7 z1 b1
    # y z9 z3 z1 b1

    y, z = graph_value(x, w)
    derivs = graph_derivatives(x, w)

    selected_derivs['dy_dz8'] = derivs['dy_dz8']
    selected_derivs['dz8_dz7'] = derivs['dz8_dz7']
    selected_derivs['dz7_dz1'] = derivs['dz7_dz1']
    selected_derivs['dz1_db1'] = derivs['dz1_db1']

    selected_derivs['dy_dz9'] = derivs['dy_dz9']
    selected_derivs['dz9_dz3'] = derivs['dz9_dz3']
    selected_derivs['dz3_dz1'] = derivs['dz3_dz1']
    selected_derivs['dz1_db1'] = derivs['dz1_db1']

    dy_db1 = derivs['dy_dz9'] * derivs['dz9_dz3'] * derivs['dz3_dz1'] * derivs['dz1_db1'] + derivs['dy_dz8'] * derivs['dz8_dz7'] * derivs['dz7_dz1'] * derivs['dz1_db1']
    # -- end --

    return selected_derivs, dy_db1


In [ ]:
answer['dy_db1'] = [dy_db1(**input) for input in inputs]


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'dy_db1'))

# Task 6
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_2} y$
- calculate that derivative

In [ ]:
# Edit this cell in task.ipynb.
def dy_db2(x, w):
    selected_derivs = {}
    dy_db2 = 0

    ## YOUR CODE HERE ##
    # -- start --
    # y z8 z7 z4 z2 b2
    # y z9 z6 z5 z2 b2

    y, z = graph_value(x, w)
    derivs = graph_derivatives(x, w)

    selected_derivs['dy_dz8'] = derivs['dy_dz8']
    selected_derivs['dz8_dz7'] = derivs['dz8_dz7']
    selected_derivs['dz7_dz4'] = derivs['dz7_dz4']
    selected_derivs['dz4_dz2'] = derivs['dz4_dz2']
    selected_derivs['dz2_db2'] = derivs['dz2_db2']

    selected_derivs['dy_dz9'] = derivs['dy_dz9']
    selected_derivs['dz9_dz6'] = derivs['dz9_dz6']
    selected_derivs['dz6_dz5'] = derivs['dz6_dz5']
    selected_derivs['dz5_dz2'] = derivs['dz5_dz2']
    selected_derivs['dz2_db2'] = derivs['dz2_db2']

    dy_db2 = derivs['dy_dz9'] * derivs['dz9_dz6'] * derivs['dz6_dz5'] * derivs['dz5_dz2'] * derivs['dz2_db2'] + derivs['dy_dz8'] * derivs['dz8_dz7'] * derivs['dz7_dz4'] * derivs['dz4_dz2'] * derivs['dz2_db2']
    # -- end --

    return selected_derivs, dy_db2


In [ ]:
answer['dy_db2'] = [dy_db2(**input) for input in inputs]


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'dy_db2'))

In [ ]:
utils.checker.save_npz(answer, str(utils.checker.get_data_dir() / 'answer.npz'))
